# Comparaison culture majoritaire : ADONIS vs RPG 2024

Pour chaque commune de la BDD ADONIS, on compare :
- `c_maj` / `cod_c_maj` : culture majoritaire selon ADONIS (IFT 2022)
- `c_maj_rpg` / `cod_c_maj_rpg` : culture majoritaire selon le RPG 2024 (surface la plus grande)

In [1]:
# Import des bibliothèques nécessaires
from pathlib import Path
import polars as pl
import geopandas as gpd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

from config import code_region_rpg
DATA_DIR = Path("../../data")
PARQUET_DIR = DATA_DIR / "parquet"

In [2]:
# Choix de la region
region = "Pays de la Loire" # ou None pour le RPG national
code = code_region_rpg[region] if region else "national"
code

'52'

## 1. Chargement des données

### Fichier ADONIS

In [3]:
# Chargement du fichier IFT ADONIS 2022
ift = pd.read_csv(
    "./../../data/raw/fre-324510908-adonis-ift-2022-v04112024.csv",
    sep=";", low_memory=False, index_col=0
)
print(ift.shape)
ift.head()

(34938, 21)


,id,nom,nom_m,insee_com,sau,sau_bio,cod_c_maj,cod_c_hbc,cod_c_h,c_maj,...,c_ift_h,ift_t,ift_t_hbc,ift_h,ift_t_hh,ift_hh_hbc,iftbc,p_bio,p_bc,p_sau
0,COMMUNE_0000000009750236,L'Abergement-Clémenciat,L'ABERGEMENT-CLEMENCIAT,01001,945.68,30.16,MIS,BTH,MIS,Maïs,...,Maïs,2.30,2.25,0.99,1.32,1.27,0.05,3.19,2.10,60.54
1,COMMUNE_0000000009750692,L'Abergement-de-Varey,L'ABERGEMENT-DE-VAREY,01002,158.10,23.93,PPH,Vigne,Vigne,Prairie permanente,...,Vigne,0.50,0.39,0.02,0.48,0.37,0.11,15.14,21.43,17.23
2,COMMUNE_0000000009750944,Ambérieu-en-Bugey,AMBERIEU-EN-BUGEY,01004,181.57,81.76,PPH,MIS,MIS,Prairie permanente,...,Maïs,0.58,0.56,0.26,0.33,0.30,0.02,45.03,4.28,7.41
3,COMMUNE_0000000009750958,Ambérieux-en-Dombes,AMBERIEUX-EN-DOMBES,01005,1017.84,NaN,MIS,BTH,MIS,Maïs,...,Maïs,2.30,2.26,1.05,1.23,1.21,0.04,NaN,1.71,63.56
4,COMMUNE_0000000009751732,Ambléon,AMBLEON,01006,125.88,29.54,PPH,BTH,BTH,Prairie permanente,...,Blé tendre,0.22,0.21,0.09,0.13,0.12,0.00,23.47,1.98,20.87


### Données spatialisées

In [4]:
# Communes
communes = gpd.read_file("./../../data/raw/ADE-COG-CARTOPLUS_4-0_GPKG_LAMB93_FRA-ED2025-01-01.gpkg", layer="COMMUNE")

In [5]:
# EPCI
epci = gpd.read_file("./../../data/raw/ADE-COG-CARTOPLUS_4-0_GPKG_LAMB93_FRA-ED2025-01-01.gpkg", layer="EPCI")

In [6]:
# Chargement du RPG 2024
rpg_pour_carte = f"../../data/raw/RPG_3-0__GPKG_LAMB93_R{code}_2024-01-01/RPG_Parcelles.gpkg"
gdf_rpg = gpd.read_file(rpg_pour_carte)

In [7]:
# Chargement du RPG 2022
gdf2022 = gpd.read_file("./../../data/raw/RPG_2-0__SHP_LAMB93_R52_2022-01-01/RPG/1_DONNEES_LIVRAISON_2023-08-01/RPG_2-0_SHP_LAMB93_R52-2022/PARCELLES_GRAPHIQUES.shp")

### Création des fichiers de travail spatialisées -> parcelles/Communes et parcelles/

Jointure spatiale RPG → communes ou  EPCI

In [8]:
# Jointure spatiale parcelles → communes
rpg_commune_2024 = gpd.sjoin(
    gdf_rpg,
    communes[["code_insee", "nom_officiel_en_majuscules", "geometry"]].rename(columns={"nom_officiel_en_majuscules": "nom_commune"}),
    how='left',
    predicate='within'
)

In [9]:
# Jointure spatiale parcelles → epci  
rpg_epci_2024 = gpd.sjoin(
    gdf_rpg,
    epci[["code_siren", "nom_officiel_en_majuscules", "geometry"]].rename(columns={"nom_officiel_en_majuscules": "nom_epci"}),
    how='left',
    predicate='within'
)

In [10]:
# Jointure spatiale parcelles → communes
rpg_commune_2022 = gpd.sjoin(
    gdf2022,
    communes[["code_insee", "nom_officiel_en_majuscules", "geometry"]].rename(columns={"nom_officiel_en_majuscules": "nom_commune"}),
    how='left',
    predicate='within')

In [11]:
# Jointure spatiale parcelles → epci
rpg_epci_2022 = gpd.sjoin(
    gdf2022,
    epci[["code_siren", "nom_officiel_en_majuscules", "geometry"]].rename(columns={"nom_officiel_en_majuscules": "nom_epci"}),
    how='left',
    predicate='within')

## 2. Culture majoritaire par commune depuis le RPG 2024

### Choix des paramètres

In [12]:
# Choix des paramètres
annee = 2022 # ou 2024
zone = "epci" # ou "epci" ou "commune"


### Création des fichiers de travail

In [13]:
# Choix des variables d'identification
col_name = f"nom_{zone}"
if zone == "commune":
    code = "code_insee"
elif zone == "epci":
    code = "code_siren"
else:
    raise ValueError("Zone non supportée")

In [14]:
# Choix du fichier à prendre en compte
file = f"rpg_{zone}_{annee}"
print(f"Fichier sélectionné : {file}")
out_file_name = f"rpg_{zone}_{annee}_df"
print("Le fichier crée sera sauvegardé sous le nom : ", out_file_name)

Fichier sélectionné : rpg_epci_2022
Le fichier crée sera sauvegardé sous le nom :  rpg_epci_2022_df


In [15]:
# Verification de l'existence du fichier
if file not in locals():
    raise ValueError(f"Le fichier {file} n'existe pas. Veuillez vérifier les choices     de paramètres.")
else: 
    print(f"Le fichier {file} est prêt à être utilisé.")
    # Selection du RPG à utiliser
    rpg = locals()[file]
    # Mettre tous les nom de colonnes en minuscules pour éviter les problèmes de casse
    rpg = rpg.rename(columns=str.lower)
    print(f"Il contient {rpg[code].nunique()} {zone}s uniques.")

Le fichier rpg_epci_2022 est prêt à être utilisé.
Il contient 95 epcis uniques.


In [16]:
# Transformation du fichier en dataframe polars pour simplifier les manipulations
rpg_pl = pl.from_pandas(rpg.drop(columns="geometry"))
rpg_pl.head()

id_parcel,surf_parc,code_cultu,code_group,culture_d1,culture_d2,index_right,code_siren,nom_epci
str,f64,str,str,str,str,f64,str,str
"""19""",0.45,"""VRC""","""21""",null,null,427.0,"""200067866""","""CC SEVRE ET LOIRE"""
"""28""",0.03,"""SNE""","""28""",null,null,705.0,"""200072700""","""CC HAUTE SARTHE ALPES MANCELLE…"
"""33""",0.73,"""VRG""","""20""",null,null,706.0,"""200072718""","""CC DE LA CHAMPAGNE CONLINOISE …"
"""67""",13.9,"""MIE""","""2""","""DRG""","""DTR""",1004.0,"""244900809""","""CC ANJOU BLEU COMMUNAUTE"""
"""81""",0.15,"""J5M""","""11""",null,null,663.0,"""200071868""","""CC DES VALLEES DU HAUT-ANJOU"""


### Calculs agrégés

In [17]:
# Calcul des surfaces totales par culture par zone dans le RPG
surf_culture_par_zone = rpg_pl.group_by(
    [   code, col_name, 
        "code_group", "code_cultu"
        ]).agg([
    pl.col("surf_parc").sum().alias("SAU_tot"),
    pl.col("surf_parc").mean().alias("SAU_moy"),
    pl.len().alias("nb_parc")
])

In [18]:
# Résumé sur les zones
df = rpg_pl.group_by([col_name, code]).agg([
    pl.col("surf_parc").sum().alias("SAU_tot"),
    pl.col("id_parcel").n_unique().alias("nb_parc"),
    pl.col("code_cultu").n_unique().alias("nb_cult"),
])

In [19]:
# Culture majoritaire = celle qui cumule le plus de surface par zone
cult_maj = (
    rpg_pl.group_by([col_name, code, "code_cultu"])
    .agg(pl.col("surf_parc").sum())
    .sort("surf_parc", descending=True)
    .group_by([col_name, code])
    .agg(pl.col("code_cultu").first().alias("cult_maj"))
)

df = df.join(cult_maj, on=[col_name, code])

In [20]:
# Un peu plus de détails en prenants les 3 cultures les plus présentes
cult_stats = (
    rpg_pl.group_by([col_name, code, "code_cultu"]).agg(
        pl.col("surf_parc").sum().alias("SAU_cult")
    )
    .with_columns(
        pl.col("SAU_cult")
          .sum()
          .over(col_name)
          .alias("SAU_tot")
    )
    .with_columns(
        (pl.col("SAU_cult") / pl.col("SAU_tot") * 100).alias("pct_SAU")
    )
    .sort([col_name, "SAU_cult"], descending=[False, True])
    .with_columns(
        pl.col("SAU_cult").rank(method="ordinal", descending=True)
          .over(col_name)
          .alias("rank")
    )
    .filter(pl.col("rank") <= 3)
)


In [21]:
# Bilan par zone et année
# 3 cultures les plus présentes par zone, avec leur surface et leur pourcentage de la SAU totale
df = df.join(
    cult_stats.pivot(
            on="rank",
            index=col_name,
            values=["code_cultu", "SAU_cult", "pct_SAU"]
        ),
        on=col_name,
        how="left"
    )

In [22]:
#Sauvegarde du dataframe intermédiaire pour éviter de refaire les calculs à chaque fois
df.write_parquet(PARQUET_DIR / f"{out_file_name}.parquet")

### Vérification de la cohérence des données

In [23]:
# Visualisation rapide
df.head()

nom_epci,code_siren,SAU_tot,nb_parc,nb_cult,cult_maj,code_cultu_1,code_cultu_2,code_cultu_3,SAU_cult_1,SAU_cult_2,SAU_cult_3,pct_SAU_1,pct_SAU_2,pct_SAU_3
str,str,f64,u32,u32,str,str,str,str,f64,f64,f64,f64,f64,f64
"""CC MAINE SAOSNOIS""","""200072676""",39938.86,10466,68,"""BTH""","""BTH""","""PPH""","""MIS""",10948.56,8521.97,6256.9,27.413301,21.337539,15.666196
"""CC DU PAYS DE LA CHATAIGNERAIE""","""248500415""",24477.71,8930,69,"""BTH""","""BTH""","""PPH""","""MIE""",5560.66,4576.79,2421.77,22.717239,18.697787,9.893777
"""CC LE GESNOIS BILURIEN""","""200072684""",17915.01,5560,63,"""PPH""","""PPH""","""BTH""","""MIS""",4069.63,3632.4,2601.09,22.716314,20.275735,14.519054
"""CC MAYENNE COMMUNAUTE""","""200055887""",43874.68,18716,58,"""MIE""","""MIE""","""PPH""","""BTH""",11048.58,9288.31,8522.77,25.182132,21.170092,19.425258
"""CC DU PAYS DE MORTAGNE""","""248500662""",16154.23,6109,81,"""PPH""","""PPH""","""MIE""","""BTH""",5669.01,1790.23,1752.05,35.093038,11.082113,10.845766


In [24]:
# Verification de la cohérence des données
nom = "SAINT-PATERNE - LE CHEVAIN"
display(surf_culture_par_zone.filter(pl.col(col_name) == nom).sort("SAU_tot", descending=True))
print("Nombre de parcelles: ", surf_culture_par_zone.filter(pl.col(col_name) == nom).select(pl.col("nb_parc").sum()).item())
print("Surface totale: ", surf_culture_par_zone.filter(pl.col(col_name) == nom).select(pl.col("SAU_tot").sum()).item())

display(df.filter(pl.col(col_name) == nom) )

code_siren,nom_epci,code_group,code_cultu,SAU_tot,SAU_moy,nb_parc
str,str,str,str,f64,f64,u32


Nombre de parcelles:  0
Surface totale:  0.0


nom_epci,code_siren,SAU_tot,nb_parc,nb_cult,cult_maj,code_cultu_1,code_cultu_2,code_cultu_3,SAU_cult_1,SAU_cult_2,SAU_cult_3,pct_SAU_1,pct_SAU_2,pct_SAU_3
str,str,f64,u32,u32,str,str,str,str,f64,f64,f64,f64,f64,f64


## 3. Agrégation des données d'IFT

In [25]:
# Agrégation des communes et epci
communes_with_epci = epci[[ "nom_officiel", "nature", "code_siren"]].rename(columns={"nom_officiel": "nom_epci"}).merge(
    communes[["nom_officiel_en_majuscules", "code_insee", "code_postal", "codes_siren_des_epci"]].rename(columns={"nom_officiel_en_majuscules": "nom_commune"}),
    left_on="code_siren",
    right_on="codes_siren_des_epci",
    how="left"
)
communes_with_epci.head()

,nom_epci,nature,code_siren,nom_commune,code_insee,code_postal,codes_siren_des_epci
0,CC Faucigny-Glières,Communauté de communes,200000172,AYSE,74024,74130,200000172
1,CC Faucigny-Glières,Communauté de communes,200000172,BONNEVILLE,74042,74130,200000172
2,CC Faucigny-Glières,Communauté de communes,200000172,BRIZON,74049,74130,200000172
3,CC Faucigny-Glières,Communauté de communes,200000172,CONTAMINE-SUR-ARVE,74087,74130,200000172
4,CC Faucigny-Glières,Communauté de communes,200000172,MARIGNIER,74164,74970,200000172


In [26]:
# Verification que qu'il y a exactement un code postal par EPCI et que tous les codes postaux sont uniques
epci_postal = communes_with_epci.groupby("codes_siren_des_epci").agg({
    "nom_epci": lambda x: x.unique(),
    "nom_epci": lambda x: x.nunique(),
    "code_postal": lambda x: x.nunique(),
    "code_insee": lambda x: x.nunique(), 
    "code_siren": lambda x: x.nunique()
}).rename(columns={
    "nom_epci": "nom_epci",
    "nom_epci": "nb_nom_epci",
    "code_postal": "nb_code_postal", "code_insee": "nb_communes", "code_siren": "nb_epci"})
display(epci_postal)

,nb_nom_epci,nb_code_postal,nb_communes,nb_epci
codes_siren_des_epci,,,,
200000172,1,2,7,1
200000438,1,3,9,1
200000545,1,2,6,1
200000628,1,4,5,1
200000800,1,1,6,1
...,...,...,...,...
249740085,1,4,4,1
249740093,1,6,6,1
249740101,1,5,5,1


In [27]:
# Jointure IFT avec EPCI 
ift_with_epci = ift.merge(
    communes_with_epci, left_on="insee_com", right_on="code_insee", how="left"
)

In [28]:
ift_with_epci.head()

,id,nom,nom_m,insee_com,sau,sau_bio,cod_c_maj,cod_c_hbc,cod_c_h,c_maj,...,p_bio,p_bc,p_sau,nom_epci,nature,code_siren,nom_commune,code_insee,code_postal,codes_siren_des_epci
0,COMMUNE_0000000009750236,L'Abergement-Clémenciat,L'ABERGEMENT-CLEMENCIAT,01001,945.68,30.16,MIS,BTH,MIS,Maïs,...,3.19,2.10,60.54,CC de la Dombes,Communauté de communes,200069193,L'ABERGEMENT-CLEMENCIAT,01001,01400,200069193
1,COMMUNE_0000000009750692,L'Abergement-de-Varey,L'ABERGEMENT-DE-VAREY,01002,158.10,23.93,PPH,Vigne,Vigne,Prairie permanente,...,15.14,21.43,17.23,CC de la Plaine de l'Ain,Communauté de communes,240100883,L'ABERGEMENT-DE-VAREY,01002,01640,240100883
2,COMMUNE_0000000009750944,Ambérieu-en-Bugey,AMBERIEU-EN-BUGEY,01004,181.57,81.76,PPH,MIS,MIS,Prairie permanente,...,45.03,4.28,7.41,CC de la Plaine de l'Ain,Communauté de communes,240100883,AMBERIEU-EN-BUGEY,01004,01500,240100883
3,COMMUNE_0000000009750958,Ambérieux-en-Dombes,AMBERIEUX-EN-DOMBES,01005,1017.84,NaN,MIS,BTH,MIS,Maïs,...,NaN,1.71,63.56,CC Dombes Saône Vallée,Communauté de communes,200042497,AMBERIEUX-EN-DOMBES,01005,01330,200042497
4,COMMUNE_0000000009751732,Ambléon,AMBLEON,01006,125.88,29.54,PPH,BTH,BTH,Prairie permanente,...,23.47,1.98,20.87,CC Bugey Sud,Communauté de communes,200040350,AMBLEON,01006,01300,200040350


In [29]:
# Nombre d'epcis uniques dans l'IFT
print("Nombre d'EPCI: ", ift_with_epci["codes_siren_des_epci"].nunique())
ift_epci = ift_with_epci.groupby(["codes_siren_des_epci", "nom_epci"]).agg(
    nb_communes=("insee_com", "nunique")
).reset_index()
ift_epci.sort_values("nb_communes", ascending=False).head()


Nombre d'EPCI:  1256


,codes_siren_des_epci,nom_epci,nb_communes
364,200067106,CA du Pays Basque,158
374,200067213,CU du Grand Reims,143
373,200067205,CA du Cotentin,129
201,200041523,CC de la Haute Saintonge,129
1036,245701206,CC du Saulnois,128


In [30]:
# Identification des données manquantes
ift_with_epci.isna().sum()

id                         0
nom                        0
nom_m                      0
insee_com                  0
sau                      333
sau_bio                 9315
cod_c_maj                333
cod_c_hbc                333
cod_c_h                    0
c_maj                      0
c_ift_hbc                  0
c_ift_h                    0
ift_t                    333
ift_t_hbc                333
ift_h                    333
ift_t_hh                 333
ift_hh_hbc               333
iftbc                    333
p_bio                   9315
p_bc                    2093
p_sau                    333
nom_epci                 197
nature                   197
code_siren               197
nom_commune              197
code_insee               197
code_postal              200
codes_siren_des_epci     197
dtype: int64

In [31]:
# Agrégation à la maille EPCI
ift_epci = pl.DataFrame(ift_with_epci)
ift_epci = ift_epci.group_by("codes_siren_des_epci","nom_epci").agg(
    pl.col("code_postal").n_unique().alias("nb_code_postal"),
    pl.col("insee_com").n_unique().alias("nb_communes"), 
    pl.col("sau").sum().alias("SAU_tot"),
    pl.col("sau_bio").sum().alias("SAU_bio_tot"),
    pl.col("ift_t_hbc").mean().alias("ift_t_hbc_moy"), 
    pl.col("ift_hh_hbc").mean().alias("ift_hh_hbc_moy"), 
    pl.col("ift_h").mean().alias("ift_h_moy"), 
)

In [32]:
ift_epci.sort("nb_communes", descending=True).head()

codes_siren_des_epci,nom_epci,nb_code_postal,nb_communes,SAU_tot,SAU_bio_tot,ift_t_hbc_moy,ift_hh_hbc_moy,ift_h_moy
str,str,u32,u32,f64,f64,f64,f64,f64
null,null,1,197,56555.76,5504.74,1.99697,1.297071,0.700808
"""200067106""","""CA du Pays Basque""",24,158,169334.42,8130.08,0.449241,0.244051,0.20557
"""200067213""","""CU du Grand Reims""",19,143,94429.34,4504.29,5.297622,3.927972,1.369371
"""200067205""","""CA du Cotentin""",19,129,89151.02,8985.75,1.347597,0.802326,0.545659
"""200041523""","""CC de la Haute Saintonge""",10,129,91465.66,3189.74,5.666357,4.828295,0.83814


In [33]:
# Culture majoritaire = celle qui cumule le plus de SAU par EPCI
cult_maj_epci = (
    pl.from_pandas(ift_with_epci[["codes_siren_des_epci", "nom_epci","c_maj", "cod_c_maj", "sau","insee_com"]])
    .group_by(["codes_siren_des_epci", "nom_epci", "c_maj", "cod_c_maj"])
    .agg(
        pl.col("sau").sum().alias("proxy_sau_culture"),
        pl.col("insee_com").len().alias("nb_communes"))
    .sort("proxy_sau_culture", descending=True))

cult_maj_epci.head()


codes_siren_des_epci,nom_epci,c_maj,cod_c_maj,proxy_sau_culture,nb_communes
str,str,str,str,f64,u32
"""200071587""","""CC Loches Sud Touraine""","""Blé tendre""","""BTH""",112398.18,64
"""200067106""","""CA du Pays Basque""","""Prairie permanente""","""PPH""",104142.06,110
"""200066660""","""CC Saint-Flour Communauté""","""Prairie permanente""","""PPH""",91231.65,52
"""200069755""","""CC Mellois en Poitou""","""Blé tendre""","""BTH""",88981.51,55
"""200067205""","""CA du Cotentin""","""Prairie permanente""","""PPH""",84446.72,119


In [34]:
# Top 3 cultures par SAU pour chaque EPCI
cult_stats_epci = cult_maj_epci.with_columns(
        pl.col("proxy_sau_culture")
          .rank(method="ordinal", descending=True)
          .over("codes_siren_des_epci")
          .alias("rank")
    ).filter(pl.col("rank") <= 3)

cult_stats_epci

codes_siren_des_epci,nom_epci,c_maj,cod_c_maj,proxy_sau_culture,nb_communes,rank
str,str,str,str,f64,u32,u32
"""200071587""","""CC Loches Sud Touraine""","""Blé tendre""","""BTH""",112398.18,64,1
"""200067106""","""CA du Pays Basque""","""Prairie permanente""","""PPH""",104142.06,110,1
"""200066660""","""CC Saint-Flour Communauté""","""Prairie permanente""","""PPH""",91231.65,52,1
"""200069755""","""CC Mellois en Poitou""","""Blé tendre""","""BTH""",88981.51,55,1
"""200067205""","""CA du Cotentin""","""Prairie permanente""","""PPH""",84446.72,119,1
…,…,…,…,…,…,…
"""245400171""","""CC Moselle et Madon""","""Aucune""",null,0.0,1,3
"""248300543""","""Métropole Toulon-Provence-Médi…","""Aucune""",null,0.0,3,3
"""243300563""","""CA Bassin d'Arcachon Sud (COBA…","""Aucune""",null,0.0,1,3


In [37]:
# Pivot → colonnes c_maj_1, cod_c_maj_1, sau_c_maj_1, ...
cult_top3_epci = cult_stats_epci.pivot(
    on="rank",
    index=["codes_siren_des_epci", "nom_epci"],
    values=["c_maj", "cod_c_maj", "proxy_sau_culture"]
).rename({
    "c_maj_1":       "c_maj_1",
    "c_maj_2":       "c_maj_2",
    "c_maj_3":       "c_maj_3",
    "cod_c_maj_1":   "cod_c_maj_1",
    "cod_c_maj_2":   "cod_c_maj_2",
    "cod_c_maj_3":   "cod_c_maj_3",
    "proxy_sau_culture_1": "sau_c_maj_1",
    "proxy_sau_culture_2": "sau_c_maj_2",
    "proxy_sau_culture_3": "sau_c_maj_3",
})

cult_top3_epci.head()

codes_siren_des_epci,nom_epci,c_maj_1,c_maj_2,c_maj_3,cod_c_maj_1,cod_c_maj_2,cod_c_maj_3,sau_c_maj_1,sau_c_maj_2,sau_c_maj_3
str,str,str,str,str,str,str,str,f64,f64,f64
"""200071587""","""CC Loches Sud Touraine""","""Blé tendre""","""Prairie en rotation longue (6 …","""Prairie permanente""","""BTH""","""PRL""","""PPH""",112398.18,3001.93,1015.74
"""200067106""","""CA du Pays Basque""","""Prairie permanente""","""Surface pastorale (SPH)""","""Surface pastorale (SPL)""","""PPH""","""SPH""","""SPL""",104142.06,39417.59,18164.67
"""200066660""","""CC Saint-Flour Communauté""","""Prairie permanente""","""Prairie en rotation longue (6 …",null,"""PPH""","""PRL""",null,91231.65,536.02,null
"""200069755""","""CC Mellois en Poitou""","""Blé tendre""","""Prairie permanente""",null,"""BTH""","""PPH""",null,88981.51,3283.69,null
"""200067205""","""CA du Cotentin""","""Prairie permanente""","""Chou""","""Maïs ensilage""","""PPH""","""CHU""","""MIE""",84446.72,2490.41,1729.79


In [38]:
# Sauvegarde du dataframe intermédiaire pour éviter de refaire les calculs à chaque fois
cult_top3_epci.write_parquet(PARQUET_DIR / "ift_cult_top3_epci.parquet")